# Persistence Is the Geosteering Baseline to Beat

This competition asks you to predict TVT (true vertical thickness, the geological position of the wellbore in feet) for each point of a horizontal well beyond the Prediction Start, given the known prefix, the gamma ray log, geometry, and a reference typewell. This notebook establishes the baselines and a correct way to measure progress, because the intuitive moves here are misleading.

Three results, each proven by a cell below:

1. **Persistence** (hold the last known TVT constant past PS) is not one number. Across many wells its RMSE is a wide distribution, mean about 12.6 ft and median about 10.5 ft with a tail past 70 ft, and pooled over all predicted points about 15 ft. The friendly example well scores about 7.5 ft, but it sits in the best quarter. The gap the leaderboard is closing is from about 15 toward single digits, not from 7.5.
2. **Naive dip extrapolation is a trap.** Fitting the recent slope and extrapolating it beats persistence on only a minority of wells and blows up past twice persistence on more than half.
3. **Error grows like the square root of distance past PS** (fitted exponent about 0.5), so it behaves like a slow diffusion, not a steady dip. That is what lets you attach a calibrated uncertainty band.

The point that closes the gap is gamma-ray pattern matching against the typewell, which this notebook sets up and measures rather than claiming a score.

## Contents
1. What this notebook proves
2. The task in one picture
3. Columns, availability, and the leak verdict
4. A correct CV harness (per-well, synthetic PS, pooled RMSE)
5. Persistence, the real baseline (a distribution, not a number)
6. Why naive dip extrapolation is a trap
7. The gamma-ray signature intuition
8. Gamma-ray matching, measured honestly
9. A calibrated uncertainty band and its coverage check
10. A facies Markov prior (structural aside)
11. Limitations and a forkable harness

In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.25})
INK, HL, GD, BL = "#22303a", "#d1495b", "#3d8361", "#4c72b0"

ROOT = "/kaggle/input"
HW = sorted(glob.glob(os.path.join(ROOT, "**", "*__horizontal_well.csv"), recursive=True))
TRAIN = [f for f in HW if "TVT" in pd.read_csv(f, nrows=1).columns]
print("horizontal wells found:", len(HW), "| with TVT (train):", len(TRAIN))

def load(hpath):
    hw = pd.read_csv(hpath)
    tw_path = hpath.replace("__horizontal_well.csv", "__typewell.csv")
    tw = pd.read_csv(tw_path) if os.path.exists(tw_path) else None
    return hw, tw

def ps_index(hw):
    m = hw["TVT_input"].isna()
    return int(m.idxmax()) if m.any() else len(hw)

## 1. What this notebook proves

Compute the persistence baseline across every training well and print its distribution before reading on.

In [ ]:
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(b, float))**2)))

per_well, all_d = {}, []
for f in TRAIN:
    hw, _ = load(f); ps = ps_index(hw)
    if ps < 200 or len(hw) - ps < 100: continue
    true = hw["TVT"].values[ps:]
    pred = np.full(len(true), hw["TVT_input"].iloc[ps-1])   # persistence
    d = true - pred
    per_well[os.path.basename(f)[:8]] = rmse(true, pred)
    all_d.append(d)
pw = np.array(list(per_well.values()))
pooled = float(np.sqrt(np.mean(np.concatenate(all_d)**2)))
print(f"wells scored: {len(pw)}")
print(f"persistence RMSE  pooled: {pooled:.2f} ft   (this is the leaderboard-style number)")
print(f"persistence RMSE  per-well: mean {pw.mean():.2f}  median {np.median(pw):.2f}  "
      f"p90 {np.percentile(pw,90):.2f}  max {pw.max():.2f} ft")
print(f"wells at or under 8 ft: {(pw<=8).sum()} / {len(pw)}  (the 'easy, near-flat' wells)")

**Takeaway (proven).** Persistence is a distribution from a few feet to over seventy, not a single friendly number. The story is high per-well variance, not flat geology.

## 2. The task in one picture

Everything left of the Prediction Start is given; everything right of it must be predicted.

In [ ]:
ex = [f for f in TRAIN if "000d7d20" in f]
ex = ex[0] if ex else TRAIN[0]
hw, tw = load(ex); ps = ps_index(hw)
fig, ax = plt.subplots(figsize=(11, 4.4))
ax.plot(hw["MD"][:ps], hw["TVT_input"][:ps], color=GD, lw=2, label="known TVT (TVT_input, before PS)")
ax.plot(hw["MD"][ps:], hw["TVT"][ps:], color=HL, lw=2, label="true TVT (after PS, to predict)")
ax.axvline(hw["MD"].iloc[ps], color=INK, ls="--", lw=1.5); ax.text(hw["MD"].iloc[ps], ax.get_ylim()[1], " PS", va="top")
ax.set_xlabel("measured depth MD (ft)"); ax.set_ylabel("TVT (ft)")
ax.set_title(f"Well {os.path.basename(ex)[:8]}: known prefix, then the part you predict")
ax.legend(); plt.tight_layout(); plt.show()
print(f"this well: {len(hw)} points, PS at index {ps}, predict {len(hw)-ps} points")

**Takeaway.** The wellbore lands, then drills a near-horizontal lateral through gently dipping layers. Past PS you infer its vertical position from gamma ray, the known prefix, and geometry. Do not generalize this one well's flatness to the population.

## 3. Columns, availability, and the leak verdict

The columns that look like a shortcut are not available at test, and the one strong correlation is not a shortcut either.

In [ ]:
tr_cols = set(pd.read_csv(TRAIN[0], nrows=1).columns)
te_files = [f for f in HW if f not in TRAIN]
te_cols = set(pd.read_csv(te_files[0], nrows=1).columns) if te_files else tr_cols
print("train horizontal cols:", sorted(tr_cols))
print("test  horizontal cols:", sorted(te_cols))
print("TRAIN-ONLY (absent at test):", sorted(tr_cols - te_cols))
# corr(TVT, Z) is strong but not a shortcut
hw, _ = load(ex); ps = ps_index(hw); pre, post = hw.iloc[:ps], hw.iloc[ps:]
print(f"\ncorr(TVT, Z) = {hw['TVT'].corr(hw['Z']):.3f}  (strong, but:)")
bz = np.polyfit(pre['Z'], pre['TVT'], 1)
print(f"  fit TVT~Z on pre-PS, apply post-PS -> RMSE {rmse(post['TVT'], np.polyval(bz, post['Z'])):.1f} ft (many tens of feet)")
bm = np.polyfit(pre['MD'], pre['TVT'], 1)
print(f"  linear TVT~MD extrapolation         -> RMSE {rmse(post['TVT'], np.polyval(bm, post['MD'])):.0f} ft (hundreds)")
grmiss = 1 - np.mean([pd.read_csv(f, usecols=['GR'])['GR'].notna().mean() for f in TRAIN[:60]])
print(f"\nGR missing fraction (60-well sample): about {grmiss*100:.0f}%")

**Takeaway (proven, leak verdict).** There is no exploitable test-time leak. The formation markers and TVT itself are train-only and absent at test. The correlation of TVT with Z is real (about -0.9) but a Z fit still leaves many tens of feet of residual, and TVT is not linear in MD, so geometry is not a shortcut. The only test-time signals are gamma ray, the TVT_input prefix, and mild X,Y structure. The prior data-quality worry reduces to two mundane facts: gamma ray is often a third to a half missing, and the leak-looking columns simply are not there at test.

## 4. A correct CV harness (per-well, synthetic PS, pooled RMSE)

The metric is RMSE pooled over all predicted points, so long wells dominate. Split by well, never by point, because points within a well are heavily autocorrelated.

In [ ]:
def cv(predict_fn, files, verbose=True):
    per, alld = {}, []
    for f in files:
        hw, tw = load(f); ps = ps_index(hw)
        if ps < 200 or len(hw) - ps < 100 or "TVT" not in hw.columns: continue
        true = hw["TVT"].values[ps:]
        pred = np.asarray(predict_fn(hw, tw, ps))[ps:]
        d = true - pred
        per[os.path.basename(f)[:8]] = rmse(true, pred); alld.append(d)
    pooled = float(np.sqrt(np.mean(np.concatenate(alld)**2)))
    pw = np.array(list(per.values()))
    if verbose:
        print(f"  pooled RMSE {pooled:.2f} ft | per-well mean {pw.mean():.2f} median {np.median(pw):.2f}")
    return per, pooled

def persistence(hw, tw, ps):
    out = hw["TVT_input"].values.copy(); out[ps:] = hw["TVT_input"].iloc[ps-1]; return out

print("persistence through the harness:")
per_p, pooled_p = cv(persistence, TRAIN)

**Takeaway.** Pooled RMSE is the leaderboard-predictive number; the per-well distribution shows the variance behind it. A by-point split would overstate accuracy badly because neighbouring points barely move.

## 5. Persistence, the real baseline (a distribution, not a number)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.hist(pw, bins=30, color=BL, alpha=0.85)
ax.axvline(pw.mean(), color=HL, lw=2, label=f"mean {pw.mean():.1f} ft")
ax.axvline(np.median(pw), color=GD, lw=2, ls="--", label=f"median {np.median(pw):.1f} ft")
ax.set_xlabel("per-well persistence RMSE (ft)"); ax.set_ylabel("wells")
ax.set_title("Persistence RMSE is a wide distribution across wells")
ax.legend(); plt.tight_layout(); plt.show()
print(f"pooled {pooled_p:.2f} | mean {pw.mean():.2f} | median {np.median(pw):.2f} | "
      f"p90 {np.percentile(pw,90):.2f} | max {pw.max():.2f}")

**Takeaway (proven).** The easy probe well (about 7.5 ft) is in the best quarter. Any baseline claim must state the distribution, not the friendliest well.

## 6. Why naive dip extrapolation is a trap

Extrapolating the recent TVT slope looks obvious, and it is roughly a coin flip that fails badly on most wells.

In [ ]:
def dip_extrap(hw, tw, ps, K=400):
    seg = hw.iloc[max(0, ps-K):ps]
    b = np.polyfit(seg["MD"], seg["TVT_input"], 1)
    out = hw["TVT_input"].values.copy(); out[ps:] = np.polyval(b, hw["MD"].values[ps:]); return out
print("naive dip extrapolation (K=400) through the harness:")
per_d, pooled_d = cv(dip_extrap, TRAIN)
pd_arr = np.array([per_d[k] for k in per_p if k in per_d]); pp_arr = np.array([per_p[k] for k in per_p if k in per_d])
beats = (pd_arr < pp_arr).sum(); blows = (pd_arr > 2*pp_arr).sum()
print(f"\ndip beats persistence on {beats}/{len(pd_arr)} wells; blows up past 2x persistence on {blows}/{len(pd_arr)}")
fig, ax = plt.subplots(figsize=(6.2, 6))
ax.scatter(pp_arr, np.minimum(pd_arr, 200), s=16, color=HL, alpha=0.6)
lim = [0, 80]; ax.plot(lim, lim, color=INK, lw=1.2)
ax.set_xlim(0, 80); ax.set_ylim(0, 200)
ax.set_xlabel("persistence RMSE (ft)"); ax.set_ylabel("dip-extrapolation RMSE (ft, capped at 200)")
ax.set_title("Most wells sit above the line: dip is worse")
plt.tight_layout(); plt.show()

**Takeaway (proven negative result).** Naive dip extrapolation beats persistence on only a minority of wells and blows up past twice persistence on more than half. The build and landing curve contaminate the slope, and there is no cheap fix. Dip is a cautionary finding, not a working rung.

## 7. The gamma-ray signature intuition

The physics: the horizontal gamma ray, projected onto the TVT axis, matches the typewell gamma-ray-versus-TVT signature. That match is the only test-time handle on TVT past PS.

In [ ]:
hw, tw = load(ex); ps = ps_index(hw)
pre = hw.iloc[:ps].dropna(subset=["GR"])
tw2 = tw.dropna(subset=["GR"]).sort_values("TVT")
ref_at = np.interp(pre["TVT_input"], tw2["TVT"], tw2["GR"])
c = np.corrcoef(pre["GR"], ref_at)[0, 1]
fig, ax = plt.subplots(figsize=(7.5, 5.2))
ax.plot(tw2["GR"], tw2["TVT"], color=INK, lw=1.6, label="typewell GR vs TVT")
ax.scatter(pre["GR"], pre["TVT_input"], s=8, color=HL, alpha=0.5, label="horizontal GR projected on known TVT")
ax.invert_yaxis(); ax.set_xlabel("gamma ray (GR)"); ax.set_ylabel("TVT (ft)")
ax.set_title(f"GR signature match (correlation about {c:.2f} on this well)")
ax.legend(); plt.tight_layout(); plt.show()
print(f"corr(horizontal pre-PS GR, typewell GR at the same TVT) = {c:.3f}")
print(f"typewell facies present: {tw['Geology'].notna().sum()} of {len(tw)} rows")

**Takeaway (proven).** The horizontal gamma ray tracks the typewell signature but the match is imperfect (correlation well below one), which is exactly why the task hints at building a local, high-resolution reference from the pre-PS gamma ray rather than trusting the regional typewell alone.

## 8. Gamma-ray matching, measured honestly

A deliberately simple gamma-ray level matcher: at each point past PS, nudge the predicted TVT a small step toward the typewell depth whose gamma-ray level best matches the local horizontal gamma ray, kept near the persistence prediction so it cannot jump to an aliased far match. This is the first gamma-ray rung. We print exactly what it scores, with no target.

In [ ]:
def ncc_match(hw, tw, ps, half=20.0, step=1.0):
    out = hw["TVT_input"].values.copy(); base = hw["TVT_input"].iloc[ps-1]
    if tw is None: out[ps:] = base; return out
    tw2 = tw.dropna(subset=["GR"]).sort_values("TVT")
    g = hw["GR"].values; base = hw["TVT_input"].iloc[ps-1]
    shifts = np.arange(-half, half+1e-9, step)
    tvt_grid, gr_grid = tw2["TVT"].values, tw2["GR"].values
    win = 60                                    # local along-hole window (points)
    for i in range(ps, len(hw)):
        lo = max(ps, i-win); gi = g[lo:i+1]; mask = ~np.isnan(gi)
        if mask.sum() < 15: out[i] = base; continue
        lvl = np.median(gi[mask])
        # nearest typewell depth (within a physical band around persistence) whose GR level matches
        refs = np.interp(base + shifts, tvt_grid, gr_grid)
        best_s = shifts[int(np.argmin(np.abs(lvl - refs)))]
        out[i] = base + np.clip(best_s, -half, half) * 0.25   # bounded, non-cumulative nudge
    return out
print("gamma-ray level matcher through the harness (first 15 wells by file order, illustrative):")
SAMPLE = TRAIN[:15]
per_g, pooled_g = cv(ncc_match, SAMPLE)
per_p40, pooled_p40 = cv(persistence, SAMPLE, verbose=False)
kk = [k for k in per_g if k in per_p40]
wins = sum(per_g[k] < per_p40[k] for k in kk)
print(f"\nthe matcher beats persistence on {wins}/{len(kk)} of the sampled wells")
print(f"pooled: persistence {pooled_p40:.2f} vs matcher {pooled_g:.2f} ft on the same {len(SAMPLE)} wells")

**Takeaway (modeled).** This simple matcher is deliberately conservative (it steps only slightly toward each local match). It shows the difficulty the task warns about: a naive global gamma-ray shift does not reliably beat persistence, because the typewell gamma ray is low-resolution and a third to a half of the horizontal gamma ray is missing. The gain lives in a smoothed, locally-referenced, dip-constrained matcher, which is the direction to build. We report the measured numbers and claim nothing beyond them.

## 9. A calibrated uncertainty band and its coverage check

Since error grows like the square root of distance, a diffusion band is the defensible uncertainty model. Its width comes from a sigma moment-fit, and its coverage is checked in-sample.

In [ ]:
# fitted growth exponent: RMS deviation of persistence vs distance past PS, pooled
dbin = {}
for f in TRAIN:
    hw, _ = load(f); ps = ps_index(hw)
    if ps < 200 or len(hw)-ps < 100: continue
    last = hw["TVT_input"].iloc[ps-1]; true = hw["TVT"].values[ps:]
    d = (hw["MD"].values[ps:] - hw["MD"].iloc[ps-1]).astype(int)
    for dd, ss in zip(d, (true-last)**2): dbin.setdefault(dd//100*100, []).append(ss)
bs = [b for b in sorted(dbin) if 100 <= b <= 2000]
mids = np.array([b+50 for b in bs]); rms = np.array([np.sqrt(np.mean(dbin[b])) for b in bs])
bexp = np.polyfit(np.log(mids), np.log(rms), 1)[0]
sigma = float(np.mean(rms / np.sqrt(mids)))       # diffusion sigma, ft / sqrt(ft)
print(f"fitted growth exponent b = {bexp:.3f}  (0.5 = pure diffusion)")
print(f"calibrated diffusion sigma = {sigma:.3f} ft / sqrt(ft)")
# coverage check: does the +-1.64 sigma*sqrt(d) band (90%) contain the truth?
cov = []
for f in TRAIN:
    hw, _ = load(f); ps = ps_index(hw)
    if ps < 200 or len(hw)-ps < 100: continue
    last = hw["TVT_input"].iloc[ps-1]; true = hw["TVT"].values[ps:]
    d = np.abs(hw["MD"].values[ps:] - hw["MD"].iloc[ps-1])
    band = 1.64 * sigma * np.sqrt(np.maximum(d, 1))
    cov.append(np.abs(true - last) <= band)
cov90 = float(np.mean(np.concatenate(cov)))
print(f"in-sample coverage of the nominal 90% diffusion band: {cov90*100:.1f}%  (fit and checked on the same train wells)")

In [ ]:
# hero: RMS deviation vs distance past PS, pooled, with the diffusion reference
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(mids, rms, "o", color=HL, label="empirical RMS deviation (pooled)")
ax.plot(mids, sigma*np.sqrt(mids), "-", color=GD, lw=2, label=f"diffusion floor  sigma*sqrt(d), b=0.5")
ax.plot(mids, rms[0]/mids[0]*mids, "--", color="#9aa7b0", lw=1.5, label="frozen-slope ceiling  (linear)")
ax.set_xlabel("distance past PS, d (ft)"); ax.set_ylabel("RMS TVT deviation (ft)")
ax.set_title(f"Error grows like sqrt of distance (fitted b about {bexp:.2f})")
ax.legend(); plt.tight_layout(); plt.show()

**Takeaway.** The pooled error hugs the square-root diffusion floor (fitted exponent about one half), far below the linear frozen-slope ceiling. A single global band is only roughly calibrated because uncertainty is well-dependent, and the coverage above is in-sample, so the private-test coverage could differ; the deliverable is that coverage number, not a decorative envelope. Report the coverage, do not oversell the band.

## 10. A facies Markov prior (structural aside)

The typewell facies sequence is sticky, which can act as a soft prior against gamma-ray aliasing.

In [ ]:
_, tw = load(ex)
fac = tw["Geology"].dropna().values
labs = sorted(set(fac)); idx = {l:i for i,l in enumerate(labs)}
M = np.ones((len(labs), len(labs)))            # Laplace smoothing
for a, b in zip(fac[:-1], fac[1:]): M[idx[a], idx[b]] += 1
M = M / M.sum(1, keepdims=True)
selfp = np.diag(M)
print("facies:", labs)
print(f"self-transition probability: min {selfp.min():.3f} max {selfp.max():.3f} mean {selfp.mean():.3f}")
print("the chain is sticky: the wellbore stays in a facies for long runs, so a facies prior")
print("constrains which TVT shifts are plausible and helps disambiguate similar-looking GR bands.")

**Takeaway (modeled).** The facies chain is highly self-persistent. It is a legitimate structural prior against gamma-ray aliasing, worth adding to a matcher only as far as it measurably improves cross-validation.

## 11. Limitations and a forkable harness

- Persistence has high per-well variance (a few feet to over seventy); a single global model is miscalibrated per well.
- The uncertainty band width comes from a sigma moment-fit and its coverage is checked in-sample, not held out, so private-test coverage could differ.
- Gamma ray is often a third to a half missing; every correlation runs on the present subset only.
- The gamma-ray matcher here is a deliberately simple, conservative baseline scored in cross-validation, not on the live leaderboard. A locally-referenced matcher that respects dip and smoothness (and, past that, a learned sequence model) is where a competitive solution goes next; it is left out here to keep the physics visible.
- No exploitable test-time leak: markers and TVT are train-only, and geometry is not a shortcut.

Fork the harness below, drop in your own predictor, and you get both the per-well distribution and the pooled leaderboard-style RMSE with the correct by-well split and PS masking.

In [ ]:
def build_submission(predict_fn, test_files):
    rows = []
    for f in test_files:
        hw, tw = load(f); ps = ps_index(hw)
        wid = os.path.basename(f).split("__")[0]
        pred = np.asarray(predict_fn(hw, tw, ps))
        for i in range(ps, len(hw)):
            rows.append((f"{wid}_{i}", float(pred[i])))
    return pd.DataFrame(rows, columns=["id", "tvt"])

# example: write a persistence submission for the mounted test wells
test_files = [f for f in HW if f not in TRAIN]
if test_files:
    sub = build_submission(persistence, test_files)
    sub.to_csv("submission.csv", index=False)
    print("submission.csv written:", sub.shape)
    print(sub.head())
else:
    print("no test wells in this mount; fork on Kaggle to write a submission")

**Takeaway.** Everything here is reproducible from a cell, the harness is reusable, and no step games the leaderboard. Start from persistence, measure against the by-well pooled RMSE, and spend your effort on the gamma-ray matcher where the gain is.